# Task 3: Publish CPG Events sang Apache Kafka

Tài liệu này ghi nhận quá trình cấu hình, khởi tạo các topic và kiểm nghiệm tích hợp hệ thống publish event streaming của CPG Parser Service lên Apache Kafka broker chạy local.

## Kiến Trúc Kafka Event Streaming
- **Broker**: Apache Kafka chạy single-node ở chế độ KRaft (không ZooKeeper) trên Docker.
- **Topics**:
  - `cpg.nodes`: Chứa `NODE_UPSERT` và `NODE_DELETE`.
  - `cpg.edges`: Chứa `EDGE_UPSERT` và `EDGE_DELETE`.
  - `source.metadata`: Chứa `FILE_METADATA_UPSERT`.
  - `parser.errors`: Chứa `PARSER_ERROR` (Topic chứa các sự kiện lỗi nghiệp vụ).
- **Partition Key**: Sử dụng `file_id` làm khóa phân vùng để đảm bảo tính nhất quán phân vùng theo từng topic (per-topic partition consistency).
- **Guarantees**:
  - `acks=all` và producer có cấu hình `enable.idempotence=True`. Lưu ý rằng `acks=all` yêu cầu acknowledgement từ toàn bộ in-sync replicas. Trong môi trường Task 3 chỉ có một broker và replication factor bằng 1, nên acknowledgement chỉ đến từ broker duy nhất và không tạo broker redundancy hoặc high availability.
  - SQLite state database chỉ được commit sau khi nhận được delivery acknowledgement từ Kafka.


## 1. Thiết Lập Môi Trường & Import Thư Viện

In [1]:
import os
import subprocess
import json
import shutil
from pathlib import Path

# Resolve PROJECT_ROOT using git
PROJECT_ROOT = Path(
    subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        check=True,
        capture_output=True,
        text=True,
    ).stdout.strip()
)

print("PROJECT_ROOT:", PROJECT_ROOT)

def get_current_offsets(bootstrap_servers="localhost:9092"):
    from confluent_kafka import Consumer, TopicPartition
    import yaml
    with open(PROJECT_ROOT / "config/topics.yaml", "r") as f:
        config_data = yaml.safe_load(f)
    topics = [t["name"] for t in config_data["topics"] if t["name"] != "connector.errors"]
    partitions_count = {
        "cpg.nodes": 3,
        "cpg.edges": 3,
        "source.metadata": 1,
        "parser.errors": 1
    }
    conf = {
        "bootstrap.servers": bootstrap_servers,
        "group.id": "offset-capturer-temp",
        "auto.offset.reset": "earliest",
        "enable.auto.commit": "false",
    }
    consumer = Consumer(conf)
    offsets = {}
    for topic in topics:
        offsets[topic] = {}
        p_count = partitions_count.get(topic, 1)
        for p in range(p_count):
            tp = TopicPartition(topic, p)
            try:
                low, high = consumer.get_watermark_offsets(tp, timeout=5.0)
                offsets[topic][p] = high
            except Exception:
                offsets[topic][p] = 0
    consumer.close()
    return offsets


PROJECT_ROOT: /home/phat/AI_Project/lab04-cpg-streaming


## 2. Kiểm Trạng Thái Kafka Broker

In [2]:
# Verify that Kafka container is running and healthy
res_ps = subprocess.run(
    ["docker", "compose", "-f", str(PROJECT_ROOT / "infra/docker-compose.yml"), "ps"],
    check=True,
    capture_output=True,
    text=True
)
print(res_ps.stdout)


NAME        IMAGE                         COMMAND                  SERVICE   CREATED        STATUS                    PORTS
cpg-kafka   confluentinc/cp-kafka:7.4.0   "/etc/confluent/dock…"   kafka     15 hours ago   Up 23 minutes (healthy)   0.0.0.0:9092->9092/tcp, [::]:9092->9092/tcp



## 3. Khởi Tạo Topics Idempotent

In [3]:
# Run topic creation script
res_topics = subprocess.run(
    [str(PROJECT_ROOT / "scripts/create_topics.sh")],
    check=True,
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT)
)
print(res_topics.stdout)


Waiting for Kafka broker to be healthy...
Kafka broker is healthy.
Creating and validating topics...
[OK] Topic 'cpg.nodes' matches desired configuration.
[OK] Topic 'cpg.edges' matches desired configuration.
[OK] Topic 'source.metadata' matches desired configuration.
[OK] Topic 'parser.errors' matches desired configuration.
[OK] Topic 'connector.errors' matches desired configuration.

=== Existing Topics ===
__consumer_offsets
connector.errors
cpg.edges
cpg.nodes
parser.errors
source.metadata


[SUCCESS] All topics match desired configurations.



## 4. Reset Smoke State cho Kafka Run
Chúng ta sẽ sử dụng một SQLite state database biệt lập để theo dõi trạng thái chạy.

In [4]:
KAFKA_SMOKE_STATE = PROJECT_ROOT / "workspace/state/notebook/kafka_publish_smoke.sqlite3"
if KAFKA_SMOKE_STATE.exists():
    KAFKA_SMOKE_STATE.unlink()

print("Isolated smoke state reset.")


Isolated smoke state reset.


## 5. Publish Smoke Events Lên Kafka
Chạy Parser Service ở live mode (`--no-dry-run`) giới hạn 1 file từ repository mục tiêu.

In [5]:
import json
# 1. Capture start offsets
start_offsets_smoke = get_current_offsets()
with open(PROJECT_ROOT / "start_offsets_smoke.json", "w") as f:
    json.dump(start_offsets_smoke, f)

# 2. Run parse-repository
env = {**os.environ, "PARSER_STATE_DB": str(KAFKA_SMOKE_STATE)}
res_smoke = subprocess.run(
    [
        "uv", "run", "lab04", "parse-repository",
        "--scope", "smoke",
        "--limit", "1",
        "--no-dry-run"
    ],
    check=True,
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
    env=env
)
print(res_smoke.stdout)

# 3. Capture end offsets
end_offsets_smoke = get_current_offsets()
with open(PROJECT_ROOT / "end_offsets_smoke.json", "w") as f:
    json.dump(end_offsets_smoke, f)


Parsing repository (scope=smoke, limit=1, dry_run=False)...
Repository run completed. Summary:
{'discovered': 2779, 'eligible': 1, 'processed': 1, 'skipped_unchanged': 0, 'failed': 0, 'node_events': 1971, 'edge_events': 2417, 'metadata_events': 1, 'error_events': 0, 'duration_ms': 55944}



## 6. Inspect & Validate Messages từ Kafka
Tiến hành consume các message vừa được đẩy lên, kiểm tra partition key (`file_id`) và validate cấu trúc bằng JSON Schema.

In [6]:
# Query expected file_id from state db
import sqlite3
conn = sqlite3.connect(KAFKA_SMOKE_STATE)
cursor = conn.cursor()
cursor.execute("SELECT file_id FROM file_state LIMIT 1")
row = cursor.fetchone()
conn.close()
expected_file_id = row[0] if row else "unknown"

res_inspect = subprocess.run(
    [
        "uv", "run", "python", "scripts/inspect_kafka_events.py",
        "--start-offsets", "start_offsets_smoke.json",
        "--end-offsets", "end_offsets_smoke.json",
        "--expected-file-id", expected_file_id
    ],
    check=True,
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
    env={**os.environ, "KAFKA_BOOTSTRAP_SERVERS": "localhost:9092"}
)
print(res_inspect.stdout)


Assigning specific partitions and seeking to offsets: [TopicPartition{topic=cpg.nodes,partition=0,offset=0,leader_epoch=None,error=None}, TopicPartition{topic=cpg.nodes,partition=1,offset=20,leader_epoch=None,error=None}, TopicPartition{topic=cpg.nodes,partition=2,offset=11836,leader_epoch=None,error=None}, TopicPartition{topic=cpg.edges,partition=0,offset=0,leader_epoch=None,error=None}, TopicPartition{topic=cpg.edges,partition=1,offset=0,leader_epoch=None,error=None}, TopicPartition{topic=cpg.edges,partition=2,offset=14506,leader_epoch=None,error=None}, TopicPartition{topic=source.metadata,partition=0,offset=10,leader_epoch=None,error=None}, TopicPartition{topic=parser.errors,partition=0,offset=7,leader_epoch=None,error=None}]
Listening for messages... (will auto-stop after 5s of inactivity)
[cpg.edges] Part:2 Off:14506 Key:526ba644702927c3bd46145a2903e7622db08269cc30575f09aa80925f5619bc Event:EDGE_UPSERT
  [OK] Schema validation passed.
[cpg.edges] Part:2 Off:14507 Key:526ba64470292

## 7. Kiểm Nghiệm Flow Lỗi Cú Pháp (Parser Error Event)
Chúng ta sẽ phân tích một tệp tin Python chứa lỗi cú pháp để kiểm tra xem `PARSER_ERROR` event có được đẩy vào topic `parser.errors` hay không, và đảm bảo state database của file này **không được commit**.

In [7]:
# Capture offsets and parse broken_syntax.py
start_offsets_err = get_current_offsets()
with open(PROJECT_ROOT / "start_offsets_err.json", "w") as f:
    json.dump(start_offsets_err, f)

# Copy broken python syntax file to target repo
target_path = PROJECT_ROOT / "workspace/source/transformers-pr-agent/broken_syntax.py"
shutil.copy(PROJECT_ROOT / "tests/fixtures/broken_syntax.py", target_path)

# Run parser on broken syntax file
env = {**os.environ, "PARSER_STATE_DB": str(KAFKA_SMOKE_STATE)}
res_parse_err = subprocess.run(
    [
        "uv", "run", "lab04", "parse-file",
        "--file", "broken_syntax.py",
        "--no-dry-run"
    ],
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
    env=env
)
print(res_parse_err.stdout)

# Clean up broken syntax file
if target_path.exists():
    target_path.unlink()

# Capture end offsets
end_offsets_err = get_current_offsets()
with open(PROJECT_ROOT / "end_offsets_err.json", "w") as f:
    json.dump(end_offsets_err, f)


Parsing single file: broken_syntax.py
File processed. Status: FAILED, content_hash: c66248af053766c7e02b9681da497cc4e2e6041914782231a74d41e60929eea9
Error details: SyntaxError in broken_syntax.py: invalid syntax at line 3, col 10



## 8. Consume & Validate Parser Error Event
Consume từ `parser.errors` topic để lấy `PARSER_ERROR` event và validate cấu trúc.

In [8]:
# Compute expected error file_id dynamically
import sys
sys.path.append(str(PROJECT_ROOT / "src"))
from parsing.identifiers import IdentifierGenerator
expected_error_file_id = IdentifierGenerator.generate_file_id("huggingface/transformers-pr-agent", Path("broken_syntax.py"))

res_inspect_err = subprocess.run(
    [
        "uv", "run", "python", "scripts/inspect_kafka_events.py",
        "--start-offsets", "start_offsets_err.json",
        "--end-offsets", "end_offsets_err.json",
        "--expected-error-file-id", expected_error_file_id
    ],
    check=True,
    capture_output=True,
    text=True,
    cwd=str(PROJECT_ROOT),
    env={**os.environ, "KAFKA_BOOTSTRAP_SERVERS": "localhost:9092"}
)
print(res_inspect_err.stdout)


Assigning specific partitions and seeking to offsets: [TopicPartition{topic=cpg.nodes,partition=0,offset=0,leader_epoch=None,error=None}, TopicPartition{topic=cpg.nodes,partition=1,offset=20,leader_epoch=None,error=None}, TopicPartition{topic=cpg.nodes,partition=2,offset=13807,leader_epoch=None,error=None}, TopicPartition{topic=cpg.edges,partition=0,offset=0,leader_epoch=None,error=None}, TopicPartition{topic=cpg.edges,partition=1,offset=0,leader_epoch=None,error=None}, TopicPartition{topic=cpg.edges,partition=2,offset=16923,leader_epoch=None,error=None}, TopicPartition{topic=source.metadata,partition=0,offset=11,leader_epoch=None,error=None}, TopicPartition{topic=parser.errors,partition=0,offset=7,leader_epoch=None,error=None}]
Listening for messages... (will auto-stop after 5s of inactivity)
[parser.errors] Part:0 Off:7 Key:25896897c637c67394f66530446af891c3339e7f2ebf0d454f64ddb92da8479c Event:PARSER_ERROR
  [OK] Schema validation passed.

=== Verification Window ===
topic=cpg.edges 

## 9. Xác Minh Transaction Boundary
Xác minh xem state database của broken_syntax.py đã bị commit hay chưa (kết quả mong đợi là không tồn tại).

In [9]:
# Load state from state store using sqlite
import sqlite3
conn = sqlite3.connect(KAFKA_SMOKE_STATE)
cursor = conn.cursor()
cursor.execute("SELECT * FROM file_state WHERE file_path LIKE '%broken_syntax.py%'")
rows = cursor.fetchall()
conn.close()

print("Database committed rows for broken_syntax.py:", len(rows))
assert len(rows) == 0, "Error: State must not be committed for syntax error file"
print("SUCCESS: Transaction boundary verified.")


Database committed rows for broken_syntax.py: 0
SUCCESS: Transaction boundary verified.


## Ordering Guarantees & Downstream Ingestion Strategy

### 1. Cơ chế đảm bảo thứ tự hiện có (Actual Guarantees)
- **Khóa phân vùng (`file_id`)**: Đảm bảo tất cả các event có cùng `file_id` trong cùng một topic sẽ luôn đi vào **cùng một partition**. Điều này giúp giữ nguyên thứ tự xuất bản (offset order) cho các event của cùng một file trên topic đó.
- **Không có thứ tự chéo topic (No Cross-Topic Ordering)**: Vì các event được định tuyến sang các topic khác nhau (`cpg.nodes`, `cpg.edges`, `source.metadata`), Kafka **không đảm bảo** bất kỳ thứ tự phân phối nào giữa các topic này. Consumer (hoặc Kafka Connect Sink) có thể nhận tin nhắn từ `cpg.edges` trước khi nhận tin nhắn tương ứng từ `cpg.nodes`.
- **Vai trò của Metadata**: Tin nhắn `FILE_METADATA_UPSERT` được gửi sau cùng trong code của Parser Service nhưng **không** tạo thành một completion barrier ở downstream vì các tin nhắn ở các topic khác có thể đến sau hoặc được xử lý không đồng bộ.

### 2. Bảng đối chiếu khả năng bảo toàn thứ tự
| Phân loại thứ tự | Khả năng bảo toàn | Chi tiết cơ chế |
|---|---|---|
| Thứ tự trong một partition của `cpg.nodes` | **Có** | Đảm bảo bởi khóa phân vùng `file_id` |
| Thứ tự trong một partition của `cpg.edges` | **Có** | Đảm bảo bởi khóa phân vùng `file_id` |
| Thứ tự chéo giữa `cpg.nodes` và `cpg.edges` | **Không** | Các topic hoạt động độc lập và không có cơ chế đồng bộ offset |
| Thứ tự chéo giữa graph topics và `source.metadata` | **Không** | Offsets của các topic khác nhau là độc lập |
| Thứ tự toàn cục (Global ordering) cho một file | **Không** | Do dữ liệu bị chia nhỏ và định tuyến sang nhiều topic |

---

## Phân biệt Error Topics và DLQ (Error Topic Semantics)

Trong hệ thống, chúng ta phân biệt rõ rệt hai kênh xử lý lỗi (failure channels) độc lập để phục vụ cho các mục đích giám sát khác nhau:

| Topic | Phân loại (Classification) | Producer (Nguồn tạo) | Mục đích (Purpose) |
|---|---|---|---|
| `parser.errors` | Parser business error topic | Parser Service | Chứa `PARSER_ERROR` khi phân tích source file thất bại (ví dụ: lỗi cú pháp Python). |
| `connector.errors` | Kafka Connect Dead Letter Queue (DLQ) | Kafka Connect | Chứa các bản ghi bị lỗi ghi Neo4j Sink ở Task 4 (dự kiến). |

### 1. parser.errors (Business Error Topic)
- Đây **không phải** là Dead Letter Queue (DLQ).
- Sự kiện `PARSER_ERROR` được tạo chủ động bởi mã nguồn của Parser Service dựa trên nghiệp vụ phân tích tĩnh, có schema sự kiện xác thực rõ ràng.
- Các lỗi cấu trúc (schema validation failure) trên payload gốc sẽ không được đưa vào topic này để tránh gây nhiễu và đứt gãy luồng xử lý.

### 2. connector.errors (Kafka Connect DLQ)
- Đây là DLQ thực thụ phục vụ cho tầng Kafka Connect (sẽ được cấu hình và kiểm chứng ở Task 4).
- Tự động bắt các bản ghi bị từ chối hoặc gặp lỗi ghi bởi Neo4j Sink Connector để giữ an toàn dữ liệu và phục vụ audit.

---

## Reflection

- **Độ tin cậy**: Việc triển khai biên an toàn "Publish-before-Commit" đảm bảo SQLite local state store luôn đồng bộ chính xác với những gì Kafka broker đã thực sự ghi nhận.
- **Tính thực tế trong Downstream Ingestion**: Nhận thức rõ giới hạn về việc không có cross-topic ordering giúp chúng ta định hình chiến lược thiết kế cho **Task 4 (Neo4j Ingestion)**. Thay vì kỳ vọng một cách sai lầm rằng node luôn đến trước edge, Neo4j sink connector và các câu lệnh Cypher MERGE sẽ được thiết kế để tự động xử lý trường hợp cạnh đến trước node (ví dụ tạo placeholder node và làm đầy thuộc tính sau). Điều này tăng tính chịu lỗi của toàn bộ hệ thống.